In [5]:
import os
import numpy as np
import pandas as pd

FLOOR_SUMMARY = r"C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_1\test_1\approach_1_avoid_step_floor\ipa_summary_approach_1_avoid_step_floor.csv"
ELBOW_SUMMARY = r"C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_1\test_1\approach_1_elbow_step\ipa_summary_approach_1_elbow_step.csv"
OUT_DIR       = r"C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_1\test_1\approach_1_elbow_step"

BATCH_SIZES = [64, 1024, 60000]

for tag, path in [("floor", FLOOR_SUMMARY), ("elbow", ELBOW_SUMMARY)]:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"[{tag}] summary not found:\n  {path}\n"
            f"Run that method's compute notebook first, or fix the path above."
        )

floor_df = pd.read_csv(FLOOR_SUMMARY); floor_df.columns = floor_df.columns.str.strip()
elbow_df = pd.read_csv(ELBOW_SUMMARY); elbow_df.columns = elbow_df.columns.str.strip()
print("floor summary:", FLOOR_SUMMARY)
print("elbow summary:", ELBOW_SUMMARY)
print(f"Loaded {len(floor_df)} / {len(elbow_df)} pruning rows.")

floor summary: C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_1\test_1\approach_1_avoid_step_floor\ipa_summary_approach_1_avoid_step_floor.csv
elbow summary: C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_1\test_1\approach_1_elbow_step\ipa_summary_approach_1_elbow_step.csv
Loaded 19 / 19 pruning rows.


In [6]:
print("""
=== Hyperparameter Comparison ===

SHARED by both methods (step detection + empirical floor asymptote):
  BN_STEP_MIN = 100   -- Minimum BN before checking for step artifacts.
                         Steps caused by early-stopping runs pulling the
                         average upward are ignored before this point.
  STEP_THRESH = 0.01  -- |CE[i] - CE[i-1]| must exceed this to trigger
                         a step cutoff. Smaller = more sensitive to artifacts.
  TAIL_N      = 50    -- Empirical floor A = mean of the last TAIL_N data
                         points in the fit window. A is then pinned (not
                         fitted) so it reflects the true data plateau.

FLOOR method  (approach_1_avoid_step_floor):
  THRESHOLD = 0.90    -- CE_L = CE_o - 0.90 * (CE_o - A)
                         Sets the learning threshold 90% of the way from
                         CE_o (max CE) down to A (asymptote). BN_learned
                         is the first batch where the averaged CE crosses
                         this threshold. IPA = |CE_o - CE_L| / BN_learned.
                         Effect of changing THRESHOLD:
                           0.85 -> CE_L is higher -> BN_learned smaller -> higher IPA
                           0.95 -> CE_L is lower  -> BN_learned larger  -> lower  IPA
                         The choice of 0.90 is ARBITRARY — there is no
                         principled reason to prefer it over 0.85 or 0.95.

ELBOW method  (approach_1_elbow_step, this work):
  NO THRESHOLD PARAMETER. NO GRID PARAMETER.
  The elbow is found directly on the actual averaged CE data points:
    1. Start point: (BN_data[0], CE_data[0])
       End point:   (BN_data[-1], A)        <- floor asymptote
    2. Normalize both axes to [0, 1]:
         x_norm = (BN - BN_data[0]) / (BN_data[-1] - BN_data[0])
         y_norm = (CE - A)          / (CE_data[0]  - A)
    3. Reference line in normalized space: x + y = 1
       (connects the normalized start (0,1) to end (1,0))
    4. Perpendicular distance from each data point to line x+y=1:
         distance ∝ x_norm + y_norm - 1   (positive above the line)
    5. elbow = the data point with maximum distance from the reference line
    6. BN_learned = elbow BN,  CE_learned = CE at the elbow data point
    7. IPA = |CE_o - CE_learned| / BN_learned

  The elbow is always a real data point — no grid, no threshold, no snapping.
  Degenerate case (P=100%, flat curve: CE_range <= 1e-10) -> IPA = NaN.
""")


=== Hyperparameter Comparison ===

SHARED by both methods (step detection + empirical floor asymptote):
  BN_STEP_MIN = 100   -- Minimum BN before checking for step artifacts.
                         Steps caused by early-stopping runs pulling the
                         average upward are ignored before this point.
  STEP_THRESH = 0.01  -- |CE[i] - CE[i-1]| must exceed this to trigger
                         a step cutoff. Smaller = more sensitive to artifacts.
  TAIL_N      = 50    -- Empirical floor A = mean of the last TAIL_N data
                         points in the fit window. A is then pinned (not
                         fitted) so it reflects the true data plateau.

FLOOR method  (approach_1_avoid_step_floor):
  THRESHOLD = 0.90    -- CE_L = CE_o - 0.90 * (CE_o - A)
                         Sets the learning threshold 90% of the way from
                         CE_o (max CE) down to A (asymptote). BN_learned
                         is the first batch where the averaged

In [7]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

BS_COLOR = {64: "#1f77b4", 1024: "#d62728", 60000: "#2ca02c"}

fig, axes = plt.subplots(1, len(BATCH_SIZES), figsize=(6 * len(BATCH_SIZES), 5), sharex=True)
if len(BATCH_SIZES) == 1:
    axes = [axes]

for ax, bs in zip(axes, BATCH_SIZES):
    col = f"IPA_Avg_{bs}"
    f_sub = floor_df.dropna(subset=[col]) if col in floor_df.columns else floor_df.iloc[0:0]
    e_sub = elbow_df.dropna(subset=[col]) if col in elbow_df.columns else elbow_df.iloc[0:0]
    ax.plot(f_sub["P%"].values, f_sub[col].values, "o--", color="#999999", ms=5, lw=1.6,
            label="floor (threshold=0.90)")
    ax.plot(e_sub["P%"].values, e_sub[col].values, "o-",  color=BS_COLOR.get(bs, "#1f77b4"), ms=5, lw=2,
            label="elbow (distance method)")
    ax.set_title(f"BS={bs}")
    ax.set_xlabel("Pruning Percentage (%)")
    ax.grid(True, alpha=0.3)
    ax.legend(frameon=False, fontsize=10)
axes[0].set_ylabel("IPA")
fig.suptitle("IPA vs Pruning  —  floor threshold=0.90 (dashed grey) vs elbow distance method (solid)",
             fontsize=12)

out_png = os.path.join(OUT_DIR, "ipa_compare_floor_vs_elbow.png")
plt.tight_layout()
plt.savefig(out_png, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {out_png}")

Saved: C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_1\test_1\approach_1_elbow_step\ipa_compare_floor_vs_elbow.png


In [8]:
# Numeric comparison: diff = elbow - floor (positive = elbow gives higher IPA)
merged = floor_df[["P%"]].copy()
for bs in BATCH_SIZES:
    col = f"IPA_Avg_{bs}"
    if col in floor_df.columns and col in elbow_df.columns:
        merged[f"floor_{bs}"] = floor_df[col].values
        merged[f"elbow_{bs}"] = elbow_df[col].values
        merged[f"diff_{bs}"]  = elbow_df[col].values - floor_df[col].values
print(merged.to_string(index=False))

   P%  floor_64  elbow_64  diff_64  floor_1024  elbow_1024  diff_1024  floor_60000  elbow_60000  diff_60000
  0.0  0.050473  0.106723 0.056250    0.066880    0.148805   0.081924     0.073994     0.137709    0.063715
 10.0  0.046512  0.123146 0.076633    0.062390    0.127922   0.065532     0.068520     0.181917    0.113396
 20.0  0.043107  0.134170 0.091063    0.056521    0.126055   0.069533     0.063717     0.153724    0.090007
 30.0  0.039116  0.128502 0.089386    0.051599    0.159500   0.107901     0.059506     0.139396    0.079889
 40.0  0.034504  0.099184 0.064680    0.046217    0.118720   0.072502     0.051140     0.134535    0.083396
 50.0  0.029249  0.092405 0.063156    0.039093    0.096995   0.057901     0.044633     0.120865    0.076231
 60.0  0.024239  0.080798 0.056559    0.033054    0.103970   0.070916     0.036322     0.131652    0.095330
 70.0  0.019691  0.065650 0.045959    0.026618    0.086863   0.060246     0.029802     0.112515    0.082714
 80.0  0.014488  0.051681 0.